# One query → chunks + top spans

Same pipeline as `custom/app/rag_app.py`: **rewrite → multi-query search → merge/dedup → BGE rerank → LLM span extraction**.

**How to use**
1. Run the first cells once (repo path → **config** → **load RAG** → **helpers**).
2. In the last cell, set **`QUERY`** to your question (string).
3. Run that last cell again whenever you change the question.

You get: **rewritten query**, **reranked chunks** (`chunk_index`, `source_file`, `rerank_score`), and **top N span strings** with their `chunk_index`.

There is also a final lookup cell where you set `CHUNK_INDEX_LOOKUP` to print the full chunk text.

**Setup**: run from repo root `verbatim-r/` (or any cwd whose parents contain `verbatim_rag/`). **Required:** `OPENROUTER_API_KEY` in either `verbatim-r/.env` or `verbatim-r/custom/.env` (not `OPENAI_API_KEY` unless you also set that). Without it, `get_custom_service()` fails when creating the OpenAI client. **Bank**: `BANK_ID` in the config cell or `CUSTOM_BANK_ID` in `.env` (`rbi`, `bawag`, `erste`, `uni`).

In [29]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from types import SimpleNamespace


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "verbatim_rag").is_dir():
            return p
    raise RuntimeError(
        "Could not find verbatim-r repo root (no verbatim_rag/). "
        "cd to the repo root or add it to sys.path."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.chdir(REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)

REPO_ROOT = /Users/tuvshinselenge/Desktop/verbatim-r


In [30]:
import os

from dotenv import load_dotenv

repo_env = REPO_ROOT / ".env"
custom_env = REPO_ROOT / "custom" / ".env"
load_dotenv(repo_env)
load_dotenv(custom_env)

api_key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
if not api_key:
    raise RuntimeError(
        "OPENROUTER_API_KEY is missing. Add it to one of:\n"
        f"- {repo_env}\n"
        f"- {custom_env}\n"
        "Get a key at https://openrouter.ai/ — this notebook needs it for rewrite, "
        "multi-query generation, and span extraction."
    )

# Bank id from custom/data/bank_profiles.json (or CUSTOM_BANK_ID in .env)
BANK_ID = "bawag"
# BANK_ID = "bawag"  # optional: uncomment to override .env for this session

NUM_DOCS = 5  # rerank top_k
PER_QUERY_K = 20  # k per generated sub-query
TOP_SPANS = 5  # how many span strings to print

# Keep extraction mode aligned with rag_app default behavior.
EXTRACTION_MODE = "auto"

print("Loaded env from:", repo_env, "and", custom_env)
print("BANK_ID =", BANK_ID)
print("EXTRACTION_MODE =", EXTRACTION_MODE, "(rag_app-aligned)")

Loaded env from: /Users/tuvshinselenge/Desktop/verbatim-r/.env and /Users/tuvshinselenge/Desktop/verbatim-r/custom/.env
BANK_ID = bawag
EXTRACTION_MODE = auto (rag_app-aligned)


In [31]:
from custom.app.rag_app import get_custom_service
from custom.setup.bank_context import get_profile_by_id, load_bank_profiles, pick_scope_for_profile
from custom.setup.paths import resolve_custom_milvus_db_path
from custom.setup.report_scope import list_report_scopes
from custom.pipeline.retrieval import get_text_and_meta, retrieve_and_rerank

DB_PATH = resolve_custom_milvus_db_path()
profile = get_profile_by_id(BANK_ID)
if not profile:
    ids = [b.get("id") for b in load_bank_profiles()]
    raise SystemExit(f"Unknown BANK_ID={BANK_ID!r}. Known ids: {ids}")

scopes = list_report_scopes(DB_PATH)
scope = pick_scope_for_profile(profile, scopes, DB_PATH)
milvus_filter = scope.filter_expr() if scope and scope.meta_value else None

legal = str(profile.get("legal_name", ""))
short = str(profile.get("short_name", ""))

print(f"Bank: {legal} ({short})")
print(f"DB:   {DB_PATH}")
print(f"Filter: {milvus_filter!r}" if milvus_filter else "Filter: None (all chunks)")

if milvus_filter is None and BANK_ID:
    print(
        "\nWarning: no PDF matched this bank in the index. "
        "Search may hit all chunks until ingest / bank_profiles are fixed."
    )

rag = get_custom_service()
print("RAG service ready.")

2026-03-24 15:58:44,146 [WARNING][__setup_ts_by_request]: failed to get mvccTs from milvus server, use client-side ts instead (iterator.py:260)


Bank: BAWAG Group AG (BAWAG Group)
DB:   /Users/tuvshinselenge/Desktop/verbatim-r/custom/milvus_verbatim_new.db
Filter: 'metadata["source_file"] == "BAWAG.pdf"'
RAG service ready.


In [32]:
def normalize_chunk_text(t: str) -> str:
    return (t or "").replace("\ufffd", "").replace("\r", "\n")


def wrap_for_extractor(chunk) -> SimpleNamespace:
    text, meta = get_text_and_meta(chunk)
    text = normalize_chunk_text(text)
    return SimpleNamespace(
        text=text,
        metadata=meta,
        source_file=meta.get("source_file"),
        page=meta.get("page"),
    )


def flatten_top_spans_with_chunk_index(
    relevant_spans: dict, wrapped_chunks: list[SimpleNamespace], limit: int
) -> list[dict]:
    """Match rag_app _rank_and_split_spans ordering, then attach chunk_index."""
    text_to_chunk_index: dict[str, int | None] = {}
    for w in wrapped_chunks:
        meta = w.metadata or {}
        text_to_chunk_index[w.text] = meta.get("chunk_index", meta.get("chunk_number"))

    flat: list[dict] = []
    # rag_app flattens by relevant_spans dict iteration order.
    for doc_text, spans in relevant_spans.items():
        cidx = text_to_chunk_index.get(doc_text)
        for s in spans or []:
            flat.append({"chunk_index": cidx, "span": s})
            if len(flat) >= limit:
                return flat
    return flat


def run_pipeline(question: str) -> dict:
    # Keep retrieval/extraction flow aligned with custom/app/rag_app.py.
    rewritten = rag.query_rewriter.rewrite(question, legal, short)
    subqs = rag.query_generator.generate_queries(rewritten, legal, short)

    merged = []
    seen = set()
    for q in subqs:
        hits = rag.index.query(q, k=PER_QUERY_K, filter=milvus_filter)
        for h in hits:
            text, meta = get_text_and_meta(h)
            text = normalize_chunk_text(text)
            if not text:
                continue
            key = (
                meta.get("source_file") or meta.get("document_id") or "",
                meta.get("chunk_index"),
                text[:200],
            )
            if key in seen:
                continue
            seen.add(key)
            merged.append(h)

    reranked, _ = rag.reranker.rerank(rewritten, merged, top_k=NUM_DOCS)
    preds = []
    for c in reranked:
        meta = getattr(c, "metadata", {}) or {}
        preds.append((meta.get("source_file"), meta.get("chunk_index")))

    wrapped = [wrap_for_extractor(c) for c in reranked]

    prev_mode = getattr(rag.extractor, "extraction_mode", "auto")
    rag.extractor.extraction_mode = EXTRACTION_MODE
    try:
        # rag_app extracts with ORIGINAL question.
        relevant = rag.extractor.extract_spans(question, wrapped)
    finally:
        rag.extractor.extraction_mode = prev_mode

    verified_count = sum(len(v or []) for v in relevant.values())
    top_spans = flatten_top_spans_with_chunk_index(relevant, wrapped, TOP_SPANS)

    rows = []
    chunk_text_by_index = {}
    for i, c in enumerate(reranked):
        text, meta = get_text_and_meta(c)
        rs = meta.get("rerank_score") if isinstance(meta, dict) else None
        if rs is None:
            rs = getattr(c, "rerank_score", None)
        nt = normalize_chunk_text(text)
        cidx = meta.get("chunk_index", meta.get("chunk_number"))
        rows.append(
            {
                "rank": i + 1,
                "chunk_index": cidx,
                "source_file": meta.get("source_file") or meta.get("source") or "",
                "page": meta.get("page", meta.get("page_number")),
                "rerank_score": rs,
                "text_preview": nt[:240] + ("…" if len(nt) > 240 else ""),
            }
        )
        if cidx is not None:
            chunk_text_by_index[cidx] = nt

    return {
        "question": question,
        "rewritten": rewritten,
        "chunk_rows": rows,
        "chunk_text_by_index": chunk_text_by_index,
        "preds": preds,
        "top_spans": top_spans,
        "all_span_count": verified_count,
        "used_raw_fallback": False,
    }


print("run_pipeline() defined.")

run_pipeline() defined.


In [33]:
# --- Edit this string, then run this cell ---
QUERY = """
Please advise the full legal address and the country of incorporation of the contracting party providing custody and / or client money services in your jurisdiction and responding to this questionnaire. If applicable, please also provide this information for the local delegate if different from the contracting entity.
"""

q = QUERY.strip()
if not q:
    raise ValueError("Set QUERY to a non-empty question above, then run this cell.")

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    import pandas as pd
except ImportError:
    pd = None

out = run_pipeline(q)
LAST_OUT = out

print("Question:", out["question"][:500] + ("…" if len(out["question"]) > 500 else ""))
rw = out["rewritten"]
print("\nRewritten:", rw[:400] + ("…" if len(rw) > 400 else ""))
print("\nPredictions (source_file, chunk_index):", out["preds"])

if pd is not None:
    display(pd.DataFrame(out["chunk_rows"]))
else:
    for row in out["chunk_rows"]:
        print(row)

print(f"\nTop {TOP_SPANS} spans (strict verified count: {out['all_span_count']}):")
# Backward-compatible: handle old runs where top_spans was list[str].
top_span_items = []
for item in out.get("top_spans", []):
    if isinstance(item, dict):
        top_span_items.append(item)
    else:
        top_span_items.append({"chunk_index": None, "span": str(item)})

for si, item in enumerate(top_span_items, 1):
    cidx = item.get("chunk_index")
    sp = item.get("span", "")
    prev = sp[:400] + ("…" if len(sp) > 400 else "")
    print(f"  {si}) [chunk_index={cidx}] {prev}")

Extracting spans (batch mode)...
Question: Please advise the full legal address and the country of incorporation of the contracting party providing custody and / or client money services in your jurisdiction and responding to this questionnaire. If applicable, please also provide this information for the local delegate if different from the contracting entity.

Rewritten: What is the full legal address and country of incorporation of BAWAG Group AG?
What is the full legal address and country of incorporation of the local delegate providing custody or client money services?

Predictions (source_file, chunk_index): [('BAWAG.pdf', 142), ('BAWAG.pdf', 1413), ('BAWAG.pdf', 1378), ('BAWAG.pdf', 636), ('BAWAG.pdf', 111)]


,rank,chunk_index,source_file,page,rerank_score,text_preview
0,1,142,BAWAG.pdf,57,1.187050,BAWAG\tGroup\tAG\tis\tthe\t parent\t company\t...
1,2,1413,BAWAG.pdf,363,1.073350,"BAWAG\tGroup\tAG\nWiedner\tGürtel\t11,\t1100\t..."
2,3,1378,BAWAG.pdf,353,0.457285,We\t have\t performed\t a\t limited\t assuranc...
3,4,636,BAWAG.pdf,207,-1.110430,We\t have\t audited\t the\t consolidated\t fin...
4,5,111,BAWAG.pdf,42,-1.560780,"As\t of\t 31\t December\t 2024,\t BAWAG\t Grou..."



Top 5 spans (strict verified count: 5):
  1) [chunk_index=142] BAWAG P.S.K. Bank für Arbeit und Wirtschaft und Österreichische Postsparkasse Aktiengesellschaft (BAWAG	P.S.K.	 AG),	 a	 subsidiary	 of	 BAWAG	 Group	 AG,	 is	 an Austrian bank operating predominantly in Austria with additional	 activities	 in	 selected	 international	 markets.
  2) [chunk_index=142] The registered	 office	 of	 BAWAG	 Group	 AG	 is	 located	 at	 Wiedner Gürtel	11,	1100	Vienna.
  3) [chunk_index=1413] BAWAG	Group	AG
Wiedner	Gürtel	11,	1100	Vienna
  4) [chunk_index=1378] BAWAG	 Group AG,	 Vienna,	 Austria
  5) [chunk_index=636] BAWAG	Group	AG,	Vienna,	Austria


In [35]:
# --- Chunk lookup: set a chunk_index from output above ---
CHUNK_INDEX_LOOKUP = 1378  # e.g. 35

if "LAST_OUT" not in globals():
    raise RuntimeError("Run the query cell first so LAST_OUT is available.")

if CHUNK_INDEX_LOOKUP is None:
    raise ValueError("Set CHUNK_INDEX_LOOKUP to an integer chunk index, then run this cell.")

# Normalize index type
try:
    lookup = int(CHUNK_INDEX_LOOKUP)
except Exception:
    lookup = CHUNK_INDEX_LOOKUP

chunk_map = LAST_OUT.get("chunk_text_by_index", {}) or {}
text = chunk_map.get(lookup)
if text is None:
    text = chunk_map.get(str(lookup))

# Self-heal older LAST_OUT objects that don't include chunk_text_by_index.
if text is None and LAST_OUT.get("question"):
    refreshed = run_pipeline(LAST_OUT["question"])
    LAST_OUT = refreshed
    chunk_map = LAST_OUT.get("chunk_text_by_index", {}) or {}
    text = chunk_map.get(lookup)
    if text is None:
        text = chunk_map.get(str(lookup))

if not text:
    known = sorted(chunk_map.keys())
    rows_known = [r.get("chunk_index") for r in LAST_OUT.get("chunk_rows", [])]
    raise KeyError(
        f"chunk_index={lookup} not found. Available in text-map: {known}; in rows: {rows_known}. "
        "Run the query cell again, then retry lookup."
    )

row = next(
    (r for r in LAST_OUT.get("chunk_rows", []) if r.get("chunk_index") in {lookup, str(lookup)}),
    {},
)
print(
    f"chunk_index={lookup} | source={row.get('source_file', '?')} | "
    f"page={row.get('page', '?')}"
)
print("-" * 80)
print(text)


chunk_index=1378 | source=BAWAG.pdf | page=353
--------------------------------------------------------------------------------
We	 have	 performed	 a	 limited	 assurance	 engagement	 in	 the connection with the consolidated non-financial reporting pursuant to Section 267a UGB (hereafter "non-financial reporting')	 for	 the	 financial	 year	 2024	 of	 the	 BAWAG	 Group AG,	 Vienna,	 Austria	 (hereinafter	 also	 referred	 to	 as	 "BAWAG Group	AG'	or	"Company').
